# Assignment A9: Performance Tuning Deep Dive

Scenario: A PySpark job runs 4 hours; optimize it.
Tasks:
1.      Identify data skew using Spark UI and fix with salting/AQE.
2.      Replace shuffle joins with broadcast joins.
3.      Tune spark.sql.shuffle.partitions.
4.      Cache/persist intermediate DataFrames.
5.      Use explain(True) and interpret Catalyst output.
6.      Document before/after runtime, resources, and DAG.

Deliverable: Markdown report + Spark UI screenshots


## Assignment Objective

A PySpark job is experiencing poor performance due to data skew,
shuffle-heavy joins, inefficient partitioning, and repeated computation.

This notebook demonstrates performance tuning techniques using local
Apache Spark:

- Data skew identification and mitigation
- Salting
- Adaptive Query Execution (AQE)
- Broadcast joins
- Shuffle partition tuning
- Cache and persist
- Catalyst query-plan analysis using `explain(True)`
- Before vs. after performance comparison

The experiments use a synthetic transaction dataset designed to
contain significant data skew.

In [1]:
# Imports and SparkSession

import sys
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
# Create SparkSession

spark = (
    SparkSession.builder
    .appName("A9_Performance_Tuning")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 4.2.0
Spark UI: http://LAPTOP-JUUB6MEL.bbrouter:4041


In [3]:
# Record the initial configuration

print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

AQE enabled: true
Shuffle partitions: 200
Default parallelism: 8


In [4]:
# We'll treat these values as our baseline configuration
# AQE enabled: true
# Shuffle partitions: 200
# Default parallelism: 8

Create workload -- to create a dataset deliberately designed to expose performance problems

Our data will contain 10,000 transactions, with one account dominating the dataset --> That is intentional skew

In [6]:
# Create a dataset designed around the performance problem

# to deliberately create a skewed dataset - A001 will have many more transactions than other accounts.
# This simulates a "hot key" that can overload one partition by dominating the transaction data (A001 → 80% of rows)

# Number of records assigned to the heavily skewed account, Account A001 will get 8,000 transactions.
skewed_rows = 8000

# Number of records for normal accounts, Every other account will get only 500 transactions.
normal_rows_per_account = 500

# Create the heavily skewed A001 portion
skewed_df = (
    spark.range(skewed_rows) # to create a DataFrame containing numbers from 0 to 7999
    .withColumn(
        "transaction_id",
        F.concat(F.lit("T_SKEW_"), F.col("id")) # to make a unique transaction ID, join the string "T_SKEW_" with the id we generated
    )
    .withColumn(
        "account_id",
        F.lit("A001") # 8,000 rows belong to the exact same key: A001
    )
    .withColumn(
        "transaction_date",
        F.to_date(F.lit("2026-01-15")) # every transaction gets: 2026-01-15
    )
    .withColumn(
        "amount",
        (F.rand(seed=42) * 1000).cast("decimal(18,2)") # Generates a random number between approximately: 0 and 1000
    )
    .withColumn(
        "transaction_type",
        F.when(F.rand(seed=43) < 0.5, "DEBIT") # Generate DEBIT or CREDIT, approximately distributed in half
         .otherwise("CREDIT")
    )
    .select(                # to keep only the columns we need
        "transaction_id",
        "account_id",
        "transaction_date",
        "amount",
        "transaction_type"
    )
)

# Create smaller datasets for normal accounts
normal_accounts = ["A002", "A003", "A004", "A005"]

normal_dfs = []

for account in normal_accounts:
    df = (
        spark.range(normal_rows_per_account) # For each normal account, we create 500 rows
        .withColumn(
            "transaction_id",
            F.concat(
                F.lit(f"T_{account}_"),
                F.col("id")
            )
        )
        .withColumn(
            "account_id",
            F.lit(account)
        )
        .withColumn(
            "transaction_date",
            F.to_date(F.lit("2026-01-15"))
        )
        .withColumn(
            "amount",
            (F.rand(seed=100 + len(normal_dfs)) * 1000)
            .cast("decimal(18,2)")
        )
        .withColumn(
            "transaction_type",
            F.when(F.rand(seed=200 + len(normal_dfs)) < 0.5, "DEBIT")
             .otherwise("CREDIT")
        )
        .select(
            "transaction_id",
            "account_id",
            "transaction_date",
            "amount",
            "transaction_type"
        )
    )

    normal_dfs.append(df)  # Put that DataFrame into the list

# Combine all account datasets into one transaction DataFrame
transactions_df = skewed_df

for df in normal_dfs:
    transactions_df = transactions_df.unionByName(df)  # Stack the rows from two DataFrames together, matching columns by their names.

print("Total transaction rows:", transactions_df.count())



Total transaction rows: 10000


In [7]:
transactions_df.printSchema()

root
 |-- transaction_id: string (nullable = false)
 |-- account_id: string (nullable = false)
 |-- transaction_date: date (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- transaction_type: string (nullable = false)



## 1.      Identify data skew using Spark UI and fix with salting/AQE

In [8]:
# Verify the skew

skew_df = (
    transactions_df
    .groupBy("account_id")
    .count()
    .orderBy(F.desc("count"))
)

skew_df.show()

+----------+-----+
|account_id|count|
+----------+-----+
|      A001| 8000|
|      A002|  500|
|      A003|  500|
|      A004|  500|
|      A005|  500|
+----------+-----+



In [ ]:
# To Establish the baseline workload

# Baseline aggregation before optimization - we'll perform a normal aggregation by account_id
baseline_agg_df = (
    transactions_df
    .groupBy("account_id")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    )
    .orderBy("account_id")
)

baseline_agg_df.show()

# Spark needs to perform a shuffle so that records belonging to the same key can be processed together
# the partition handling A001 can end up doing substantially more work than partitions handling the other keys - That is the performance problem we want to optimize.

+----------+-----------------+------------+
|account_id|transaction_count|total_amount|
+----------+-----------------+------------+
|      A001|             8000|  3981946.73|
|      A002|              500|   261194.44|
|      A003|              500|   263532.15|
|      A004|              500|   262574.50|
|      A005|              500|   264328.06|
+----------+-----------------+------------+



In [10]:
# let's measure the baseline

import time

start_time = time.perf_counter()

baseline_agg_df.count() # to execute the aggregation

baseline_time = time.perf_counter() - start_time

print(f"Baseline runtime: {baseline_time:.4f} seconds")

Baseline runtime: 3.5668 seconds


In [11]:
# Add a recognizable description to the next Spark job
spark.sparkContext.setJobDescription("A9 Baseline Aggregation - Skew Analysis")

baseline_agg_df.count()

5

### Fix the skew with salting

- instead of initially grouping by - account_id
- we will create a temporary salt column and group by - account_id + salt
- and the we can eventually aggregate those partial results back to the original account_id

In [12]:
# To add a deterministic salt

# deterministic salt bucket from 0 to 3.
# The salt is temporary and is used only to spread a skewed key.
salted_transactions_df = (
    transactions_df
    .withColumn(
        "salt",
        F.pmod(F.monotonically_increasing_id(), 4) # generates an increasing identifier for rows
    )
)

salt_distribution_df = (
    salted_transactions_df
    .groupBy("account_id", "salt")
    .count()
    .orderBy("account_id", "salt")
)

salt_distribution_df.show()

+----------+----+-----+
|account_id|salt|count|
+----------+----+-----+
|      A001|   0| 2000|
|      A001|   1| 2000|
|      A001|   2| 2000|
|      A001|   3| 2000|
|      A002|   0|  128|
|      A002|   1|  128|
|      A002|   2|  124|
|      A002|   3|  120|
|      A003|   0|  128|
|      A003|   1|  128|
|      A003|   2|  124|
|      A003|   3|  120|
|      A004|   0|  128|
|      A004|   1|  128|
|      A004|   2|  124|
|      A004|   3|  120|
|      A005|   0|  128|
|      A005|   1|  128|
|      A005|   2|  124|
|      A005|   3|  120|
+----------+----+-----+



First-stage salted aggregation

In [13]:
# First-stage aggregation using salt
salted_agg_df = (
    salted_transactions_df
    .groupBy("account_id", "salt")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    )
)

salted_agg_df.orderBy("account_id", "salt").show()

+----------+----+-----------------+------------+
|account_id|salt|transaction_count|total_amount|
+----------+----+-----------------+------------+
|      A001|   0|             2000|  1011089.06|
|      A001|   1|             2000|   981307.26|
|      A001|   2|             2000|  1004515.63|
|      A001|   3|             2000|   985034.78|
|      A002|   0|              128|    66385.28|
|      A002|   1|              128|    63065.07|
|      A002|   2|              124|    67863.95|
|      A002|   3|              120|    63880.14|
|      A003|   0|              128|    67346.95|
|      A003|   1|              128|    64148.47|
|      A003|   2|              124|    67248.68|
|      A003|   3|              120|    64788.05|
|      A004|   0|              128|    67611.90|
|      A004|   1|              128|    63931.16|
|      A004|   2|              124|    65468.83|
|      A004|   3|              120|    65562.61|
|      A005|   0|              128|    70668.61|
|      A005|   1|   

Second-stage aggregation

In [14]:
# Combine salted partial results back to account level
final_salted_agg_df = (
    salted_agg_df
    .groupBy("account_id")
    .agg(
        F.sum("transaction_count").alias("transaction_count"),
        F.sum("total_amount").alias("total_amount")
    )
    .orderBy("account_id")
)

final_salted_agg_df.show()

+----------+-----------------+------------+
|account_id|transaction_count|total_amount|
+----------+-----------------+------------+
|      A001|             8000|  3981946.73|
|      A002|              500|   261194.44|
|      A003|              500|   263532.15|
|      A004|              500|   262574.50|
|      A005|              500|   264328.06|
+----------+-----------------+------------+



For verification — compare with baseline

In [15]:
# Compare salted result with baseline result
comparison_df = (
    baseline_agg_df.alias("base")
    .join(
        final_salted_agg_df.alias("salted"),
        "account_id"
    )
    .select(
        "account_id",
        F.col("base.transaction_count").alias("baseline_count"),
        F.col("salted.transaction_count").alias("salted_count"),
        F.col("base.total_amount").alias("baseline_amount"),
        F.col("salted.total_amount").alias("salted_amount")
    )
)

comparison_df.show()

+----------+--------------+------------+---------------+-------------+
|account_id|baseline_count|salted_count|baseline_amount|salted_amount|
+----------+--------------+------------+---------------+-------------+
|      A003|           500|         500|      263532.15|    263532.15|
|      A004|           500|         500|      262574.50|    262574.50|
|      A002|           500|         500|      261194.44|    261194.44|
|      A001|          8000|        8000|     3981946.73|   3981946.73|
|      A005|           500|         500|      264328.06|    264328.06|
+----------+--------------+------------+---------------+-------------+



To demonstrate whether salting actually improved runtime.

In [16]:
# Measure salted aggregation runtime
spark.sparkContext.setJobDescription(
    "A9 Salted Aggregation - Skew Fix"
)

start_time = time.perf_counter()

final_salted_agg_df.count()

salted_time = time.perf_counter() - start_time

print(f"Salted aggregation runtime: {salted_time:.4f} seconds")

Salted aggregation runtime: 4.1022 seconds


Note: extra aggregation stage and local execution overhead can easily outweigh the benefit of distributing skew.

### AQE

Check AQE configuration

In [17]:
# Check AQE settings
print("AQE enabled:",
      spark.conf.get("spark.sql.adaptive.enabled"))

print("AQE skew join enabled:",
      spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

print("Shuffle partitions:",
      spark.conf.get("spark.sql.shuffle.partitions"))

AQE enabled: true
AQE skew join enabled: true
Shuffle partitions: 200


Create a skewed join dataset

In [18]:
# We'll reuse transactions_df as our large/skewed side and create a small account dimension

# Small dimension table for the join
account_dimension_df = spark.createDataFrame(
    [
        ("A001", "Premium"),
        ("A002", "Standard"),
        ("A003", "Standard"),
        ("A004", "Standard"),
        ("A005", "Standard")
    ],
    ["account_id", "account_type"]
)

account_dimension_df.show()

+----------+------------+
|account_id|account_type|
+----------+------------+
|      A001|     Premium|
|      A002|    Standard|
|      A003|    Standard|
|      A004|    Standard|
|      A005|    Standard|
+----------+------------+



Force a shuffle join

- We don't want Spark deciding to broadcast the tiny dimension table automatically, because then the skewed shuffle-join behavior wouldn't be demonstrated.

In [19]:
# Force a shuffle-based join
skew_join_df = (
    transactions_df
    .hint("merge")
    .join(
        account_dimension_df.hint("merge"),
        "account_id"
    )
)

skew_join_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [account_id#2, transaction_id#1, transaction_date#3, amount#4, transaction_type#5, account_type#286]
   +- SortMergeJoin [account_id#2], [account_id#285], Inner
      :- Sort [account_id#2 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(account_id#2, 200), ENSURE_REQUIREMENTS, [plan_id=1678]
      :     +- Union
      :        :- Project [transaction_id#1, A001 AS account_id#2, 2026-01-15 AS transaction_date#3, amount#4, CASE WHEN (rand(43) < 0.5) THEN DEBIT ELSE CREDIT END AS transaction_type#5]
      :        :  +- Project [concat(T_SKEW_, cast(id#0L as string)) AS transaction_id#1, cast((rand(42) * 1000.0) as decimal(18,2)) AS amount#4]
      :        :     +- Range (0, 8000, step=1, splits=8)
      :        :- Project [transaction_id#7, A002 AS account_id#8, 2026-01-15 AS transaction_date#9, amount#10, CASE WHEN (rand(200) < 0.5) THEN DEBIT ELSE CREDIT END AS transaction_type#11]
      :        :  +

To Run with AQE OFF

In [20]:
# Disable AQE for baseline comparison
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")

print("AQE enabled:",
      spark.conf.get("spark.sql.adaptive.enabled"))

AQE enabled: false


In [21]:
# Measure shuffle join without AQE
spark.sparkContext.setJobDescription(
    "A9 Skewed Join - AQE OFF"
)

start_time = time.perf_counter()

skew_join_df.count()

aqe_off_time = time.perf_counter() - start_time

print(f"AQE OFF runtime: {aqe_off_time:.4f} seconds")

AQE OFF runtime: 16.4824 seconds


To Run with AQE ON

In [22]:
# Now we'll run the same skewed join with Adaptive Query Execution enabled and compare

# Enable AQE and skew-join handling
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("AQE enabled:",
      spark.conf.get("spark.sql.adaptive.enabled"))

print("AQE skew join enabled:",
      spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

AQE enabled: true
AQE skew join enabled: true


In [23]:
# Measure the same join with AQE enabled
spark.sparkContext.setJobDescription(
    "A9 Skewed Join - AQE ON"
)

start_time = time.perf_counter()

skew_join_df.count()

aqe_on_time = time.perf_counter() - start_time

print(f"AQE ON runtime: {aqe_on_time:.4f} seconds")

AQE ON runtime: 13.2477 seconds


AQE was enabled with skew-join handling, and the measured stage duration decreased from 13 s to 12 s. Because the test dataset is small and runs locally, the observed improvement is modest and should not be generalized to the 4-hour production scenario

### 2.      Replace shuffle joins with broadcast joins.

Establish the shuffle-join baseline

In [ ]:
# Create shuffle-join baseline
shuffle_join_df = (
    transactions_df
    .hint("merge")
    .join(
        account_dimension_df.hint("merge"), # make Spark use this shuffle-based strategy
        "account_id"
    )
)

shuffle_join_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [account_id#2, transaction_id#1, transaction_date#3, amount#4, transaction_type#5, account_type#286]
   +- SortMergeJoin [account_id#2], [account_id#285], Inner
      :- Sort [account_id#2 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(account_id#2, 200), ENSURE_REQUIREMENTS, [plan_id=2170]
      :     +- Union
      :        :- Project [transaction_id#1, A001 AS account_id#2, 2026-01-15 AS transaction_date#3, amount#4, CASE WHEN (rand(43) < 0.5) THEN DEBIT ELSE CREDIT END AS transaction_type#5]
      :        :  +- Project [concat(T_SKEW_, cast(id#0L as string)) AS transaction_id#1, cast((rand(42) * 1000.0) as decimal(18,2)) AS amount#4]
      :        :     +- Range (0, 8000, step=1, splits=8)
      :        :- Project [transaction_id#7, A002 AS account_id#8, 2026-01-15 AS transaction_date#9, amount#10, CASE WHEN (rand(200) < 0.5) THEN DEBIT ELSE CREDIT END AS transaction_type#11]
      :        :  +

- Spark is using Sort-merge join -- SortMergeJoin [account_id#2], [account_id#285], Inner
- Spark is shuffling/redistributing both datasets by account_id before performing the join -- Exchange hashpartitioning(account_id, 200)

measure the shuffle-join baseline

In [25]:
# Measure the shuffle-join baseline
spark.sparkContext.setJobDescription(
    "A9 Shuffle Join - Baseline"
)

start_time = time.perf_counter()

shuffle_join_df.count()

shuffle_join_time = time.perf_counter() - start_time

print(f"Shuffle join runtime: {shuffle_join_time:.4f} seconds")

Shuffle join runtime: 15.3626 seconds


Optimisation using Broadcast join

In [26]:
# Broadcast the small dimension table
broadcast_join_df = (
    transactions_df
    .join(
        F.broadcast(account_dimension_df),
        "account_id"
    )
)

broadcast_join_df.show(5)

+----------+--------------+----------------+------+----------------+------------+
|account_id|transaction_id|transaction_date|amount|transaction_type|account_type|
+----------+--------------+----------------+------+----------------+------------+
|      A001|      T_SKEW_0|      2026-01-15|619.19|          CREDIT|     Premium|
|      A001|      T_SKEW_1|      2026-01-15|509.60|          CREDIT|     Premium|
|      A001|      T_SKEW_2|      2026-01-15|832.53|           DEBIT|     Premium|
|      A001|      T_SKEW_3|      2026-01-15|263.23|           DEBIT|     Premium|
|      A001|      T_SKEW_4|      2026-01-15|670.29|          CREDIT|     Premium|
+----------+--------------+----------------+------+----------------+------------+
only showing top 5 rows


In [27]:
# Inspect the broadcast join plan
broadcast_join_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [account_id#2, transaction_id#1, transaction_date#3, amount#4, transaction_type#5, account_type#286]
   +- BroadcastHashJoin [account_id#2], [account_id#285], Inner, BuildRight, false, false
      :- Union
      :  :- Project [transaction_id#1, A001 AS account_id#2, 2026-01-15 AS transaction_date#3, amount#4, CASE WHEN (rand(43) < 0.5) THEN DEBIT ELSE CREDIT END AS transaction_type#5]
      :  :  +- Project [concat(T_SKEW_, cast(id#0L as string)) AS transaction_id#1, cast((rand(42) * 1000.0) as decimal(18,2)) AS amount#4]
      :  :     +- Range (0, 8000, step=1, splits=8)
      :  :- Project [transaction_id#7, A002 AS account_id#8, 2026-01-15 AS transaction_date#9, amount#10, CASE WHEN (rand(200) < 0.5) THEN DEBIT ELSE CREDIT END AS transaction_type#11]
      :  :  +- Project [concat(T_A002_, cast(id#6L as string)) AS transaction_id#7, cast((rand(100) * 1000.0) as decimal(18,2)) AS amount#10]
      :  :     +- Range (0

Spark is using Broadcast join -- BroadcastHashJoin [account_id#2], [account_id#285], Inner, BuildRight, false, false

Measure the broadcast join

In [28]:
# Measure broadcast join runtime
spark.sparkContext.setJobDescription(
    "A9 Broadcast Join - Optimized"
)

start_time = time.perf_counter()

broadcast_join_df.count()

broadcast_join_time = time.perf_counter() - start_time

print(f"Broadcast join runtime: {broadcast_join_time:.4f} seconds")

Broadcast join runtime: 13.8781 seconds


| Join strategy  |       Runtime |
| -------------- | ------------: |
| Shuffle Join   | **15.3626 s** |
| Broadcast Join | **13.8781 s** |

Broadcast join was ~9.7% faster than the shuffle-join baseline

- I replaced a forced Sort-Merge Join with a Broadcast Hash Join because the account dimension was very small. The physical plan changed from SortMergeJoin with Exchange operations to BroadcastHashJoin. On my local benchmark, runtime decreased from 15.36 s to 13.88 s, approximately a 9.7% improvement.

## 3.      Tune spark.sql.shuffle.partitions.

For shuffle operations, Spark needs to divide the shuffled data into partitions
- spark.sql.shuffle.partitions --> controls the default number of shuffle partitions for many shuffle operations

In [29]:
# Check current shuffle partition setting
print(
    "Current shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

print(
    "Default parallelism:",
    spark.sparkContext.defaultParallelism
)

Current shuffle partitions: 200
Default parallelism: 8


### Shuffle Partition Tuning

Turn AQE off for the controlled test

In [30]:
# Disable AQE for a controlled partition comparison
spark.conf.set("spark.sql.adaptive.enabled", "false")

print(
    "AQE enabled:",
    spark.conf.get("spark.sql.adaptive.enabled")
)

AQE enabled: false


- Baseline with 200 partitions

In [ ]:
# We'll use the aggregation workload because it definitely involves a shuffle

# Set baseline shuffle partition count
spark.conf.set("spark.sql.shuffle.partitions", "200")

print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Shuffle partitions: 200


In [32]:
# Measure aggregation with 200 shuffle partitions
spark.sparkContext.setJobDescription(
    "A9 Shuffle Partitions - 200"
)

start_time = time.perf_counter()

(
    transactions_df
    .groupBy("account_id")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    )
    .count()
)

partitions_200_time = time.perf_counter() - start_time

print(
    f"Runtime with 200 partitions: "
    f"{partitions_200_time:.4f} seconds"
)

Runtime with 200 partitions: 5.9353 seconds


- Test with 8 partitions

In [33]:
# Reduce shuffle partitions to match local parallelism
spark.conf.set("spark.sql.shuffle.partitions", "8")

print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Shuffle partitions: 8


In [34]:
# Measure aggregation with 8 shuffle partitions
spark.sparkContext.setJobDescription(
    "A9 Shuffle Partitions - 8"
)

start_time = time.perf_counter()

(
    transactions_df
    .groupBy("account_id")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    )
    .count()
)

partitions_8_time = time.perf_counter() - start_time

print(
    f"Runtime with 8 partitions: "
    f"{partitions_8_time:.4f} seconds"
)

Runtime with 8 partitions: 7.0099 seconds


- Test with 32 partitions

In [35]:
# Increase shuffle partitions moderately
spark.conf.set("spark.sql.shuffle.partitions", "32")

print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Shuffle partitions: 32


In [36]:
# Measure aggregation with 32 shuffle partitions
spark.sparkContext.setJobDescription(
    "A9 Shuffle Partitions - 32"
)

start_time = time.perf_counter()

(
    transactions_df
    .groupBy("account_id")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    )
    .count()
)

partitions_32_time = time.perf_counter() - start_time

print(
    f"Runtime with 32 partitions: "
    f"{partitions_32_time:.4f} seconds"
)

Runtime with 32 partitions: 5.7479 seconds


## 4.      Cache/persist intermediate DataFrames.

Turn AQE back on before continuing

In [37]:
# Restore adaptive execution
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# Restore the default shuffle partition setting
spark.conf.set("spark.sql.shuffle.partitions", "200")

print(
    "AQE:",
    spark.conf.get("spark.sql.adaptive.enabled")
)

print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

AQE: true
Shuffle partitions: 200


Create a reusable intermediate DataFrame

In [38]:
# Create an intermediate DataFrame that will be reused
filtered_transactions_df = (
    transactions_df
    .filter(F.col("amount") > 500)
    .select(
        "transaction_id",
        "account_id",
        "amount"
    )
)

filtered_transactions_df.show(5)

+--------------+----------+------+
|transaction_id|account_id|amount|
+--------------+----------+------+
|      T_SKEW_0|      A001|619.19|
|      T_SKEW_1|      A001|509.60|
|      T_SKEW_2|      A001|832.53|
|      T_SKEW_4|      A001|670.29|
|      T_SKEW_5|      A001|517.33|
+--------------+----------+------+
only showing top 5 rows


1. First measure without caching

In [39]:
# First action without caching
spark.sparkContext.setJobDescription(
    "A9 Cache - First Action Without Cache"
)

start_time = time.perf_counter()

filtered_transactions_df.count()

no_cache_first_time = time.perf_counter() - start_time

print(
    f"First action without cache: "
    f"{no_cache_first_time:.4f} seconds"
)

First action without cache: 3.3860 seconds


In [40]:
# Second action without caching
spark.sparkContext.setJobDescription(
    "A9 Cache - Second Action Without Cache"
)

start_time = time.perf_counter()

filtered_transactions_df.groupBy("account_id").count().count()

no_cache_second_time = time.perf_counter() - start_time

print(
    f"Second action without cache: "
    f"{no_cache_second_time:.4f} seconds"
)

Second action without cache: 0.9267 seconds


Persist the intermediate DataFrame

In [41]:
# We will use persist() rather than only cache() because it lets us explicitly choose a storage level.

from pyspark import StorageLevel

In [ ]:
# Persist the reusable intermediate DataFrame
filtered_transactions_df = filtered_transactions_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

# Spark will try to keep the persisted data in memory.
# If everything doesn't fit in memory, Spark can store the remaining partitions on disk rather than recomputing them.

print("DataFrame persisted.")

DataFrame persisted.


Materialize the cache

In [43]:
# Materialize the persisted DataFrame
spark.sparkContext.setJobDescription(
    "A9 Cache - Materialize Persisted DataFrame"
)

start_time = time.perf_counter()

filtered_transactions_df.count()

cache_materialization_time = time.perf_counter() - start_time

print(
    f"Cache materialization time: "
    f"{cache_materialization_time:.4f} seconds"
)

Cache materialization time: 3.4279 seconds


Reuse the persisted DataFrame

In [44]:
# First action using persisted data
spark.sparkContext.setJobDescription(
    "A9 Cache - First Reuse"
)

start_time = time.perf_counter()

filtered_transactions_df.count()

cached_first_time = time.perf_counter() - start_time

print(
    f"First action with cache: "
    f"{cached_first_time:.4f} seconds"
)

First action with cache: 3.1115 seconds


In [45]:
# Second action using persisted data
spark.sparkContext.setJobDescription(
    "A9 Cache - Second Reuse"
)

start_time = time.perf_counter()

filtered_transactions_df.groupBy("account_id").count().count()

cached_second_time = time.perf_counter() - start_time

print(
    f"Second action with cache: "
    f"{cached_second_time:.4f} seconds"
)

Second action with cache: 2.1295 seconds


| Test                     |      Runtime |
| ------------------------ | -----------: |
| First action, no cache   | **3.3860 s** |
| Second action, no cache  | **0.9267 s** |
| Cache materialization    | **3.4279 s** |
| First action with cache  | **3.1115 s** |
| Second action with cache | **2.1295 s** |

Persistence was demonstrated on a reusable intermediate DataFrame. Because the workload was small and executed locally, caching did not produce a significant runtime improvement in this benchmark. The experiment demonstrates the mechanism and the conditions under which persistence can be beneficial.

- The more expensive the lineage and the more times the result is reused, the more attractive persistence becomes.

Let's confirm Spark is actually reusing the cache

In [47]:
# Check whether the DataFrame is marked for persistence
print("Is persisted:", filtered_transactions_df.is_cached)

Is persisted: True


In [48]:
# Check whether the cached relation is used
filtered_transactions_df.explain()

== Physical Plan ==
InMemoryTableScan [transaction_id#1, account_id#2, amount#4]
   +- InMemoryRelation [transaction_id#1, account_id#2, amount#4], StorageLevel(disk, memory, 1 replicas)
         +- Union
            :- *(1) Filter (isnotnull(amount#4) AND (amount#4 > 500.00))
            :  +- *(1) Project [concat(T_SKEW_, cast(id#0L as string)) AS transaction_id#1, A001 AS account_id#2, cast((rand(42) * 1000.0) as decimal(18,2)) AS amount#4]
            :     +- *(1) Range (0, 8000, step=1, splits=8)
            :- *(2) Filter (isnotnull(amount#10) AND (amount#10 > 500.00))
            :  +- *(2) Project [concat(T_A002_, cast(id#6L as string)) AS transaction_id#7, A002 AS account_id#8, cast((rand(100) * 1000.0) as decimal(18,2)) AS amount#10]
            :     +- *(2) Range (0, 500, step=1, splits=8)
            :- *(3) Filter (isnotnull(amount#16) AND (amount#16 > 500.00))
            :  +- *(3) Project [concat(T_A003_, cast(id#12L as string)) AS transaction_id#13, A003 AS account_i

I see - "InMemoryTableScan" --> cached relation is used -- It indicates that Spark is scanning data from a cached/persisted in-memory representation rather than recomputing the original lineage from the source.

Clean up persisted Data - Cache expensive, reusable data when the benefit outweighs the storage cost, and release it when it is no longer needed.

In [49]:
# Release persisted data
filtered_transactions_df.unpersist()

print("Persisted DataFrame released.")

Persisted DataFrame released.


In [52]:
# verify clean up
print("Is persisted:", filtered_transactions_df.is_cached)

Is persisted: False


A reusable intermediate DataFrame was explicitly cached and materialized. Spark confirmed the cached state and the physical plan showed InMemoryTableScan/InMemoryRelation. Because the test workload was small and local, caching did not demonstrate a large performance gain; the experiment primarily demonstrated when and how persistence is used

## 5.      Use explain(True) and interpret Catalyst output.


In [64]:
# Show all Catalyst planning stages
broadcast_join_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [account_id])
:- Union false, false
:  :- Project [transaction_id#1, account_id#2, transaction_date#3, amount#4, transaction_type#5]
:  :  +- Project [id#0L, transaction_id#1, account_id#2, transaction_date#3, amount#4, CASE WHEN (rand(43) < 0.5) THEN DEBIT ELSE CREDIT END AS transaction_type#5]
:  :     +- Project [id#0L, transaction_id#1, account_id#2, transaction_date#3, cast((rand(42) * cast(1000 as double)) as decimal(18,2)) AS amount#4]
:  :        +- Project [id#0L, transaction_id#1, account_id#2, to_date(2026-01-15, None, Some(Asia/Calcutta), true) AS transaction_date#3]
:  :           +- Project [id#0L, transaction_id#1, A001 AS account_id#2]
:  :              +- Project [id#0L, concat(T_SKEW_, cast(id#0L as string)) AS transaction_id#1]
:  :                 +- Range (0, 8000, step=1, splits=Some(8))
:  :- Project [transaction_id#7, account_id#8, transaction_date#9, amount#10, transaction_type#11]
:  :  +- Project [transaction_i

Lets use a Simple Catalyst Example- creatiing a tiny DataFrame with just a few columns and apply:
- filter
- calculation
- selection


In [65]:
# Create a small example DataFrame
simple_df = spark.createDataFrame(
    [
        ("A001", 100),
        ("A002", 600),
        ("A003", 800),
        ("A004", 300)
    ],
    ["account_id", "amount"]
)

simple_df.show()

+----------+------+
|account_id|amount|
+----------+------+
|      A001|   100|
|      A002|   600|
|      A003|   800|
|      A004|   300|
+----------+------+



In [66]:
# Filter and calculate a new column
catalyst_df = (
    simple_df
    .filter(F.col("amount") > 500)
    .withColumn("tax", F.col("amount") * 0.10)
    .select("account_id", "tax")
)

catalyst_df.show()

+----------+----+
|account_id| tax|
+----------+----+
|      A002|60.0|
|      A003|80.0|
+----------+----+



In [67]:
# Show all Catalyst planning stages
catalyst_df.explain(True)

== Parsed Logical Plan ==
'Project ['account_id, 'tax]
+- Project [account_id#893, amount#894L, (cast(amount#894L as double) * 0.1) AS tax#902]
   +- Filter (amount#894L > cast(500 as bigint))
      +- LogicalRDD [account_id#893, amount#894L], false

== Analyzed Logical Plan ==
account_id: string, tax: double
Project [account_id#893, tax#902]
+- Project [account_id#893, amount#894L, (cast(amount#894L as double) * 0.1) AS tax#902]
   +- Filter (amount#894L > cast(500 as bigint))
      +- LogicalRDD [account_id#893, amount#894L], false

== Optimized Logical Plan ==
Project [account_id#893, (cast(amount#894L as double) * 0.1) AS tax#902]
+- Filter (isnotnull(amount#894L) AND (amount#894L > 500))
   +- LogicalRDD [account_id#893, amount#894L], false

== Physical Plan ==
*(1) Project [account_id#893, (cast(amount#894L as double) * 0.1) AS tax#902]
+- *(1) Filter (isnotnull(amount#894L) AND (amount#894L > 500))
   +- *(1) Scan ExistingRDD[account_id#893,amount#894L]



### Catalyst Plan Interpretation

| Plan Stage | Purpose |
|---|---|
| **Parsed Logical Plan** | Shows how Spark initially interprets the query. |
| **Analyzed Logical Plan** | Validates the query by resolving columns, data types, and references. |
| **Optimized Logical Plan** | Applies Catalyst optimization rules to simplify and improve the logical plan. |
| **Physical Plan** | Selects the actual execution operators Spark will use to run the query. |

#### Interpretation of the Output

- **Parsed Logical Plan:** Spark represents the requested `filter`, calculated `tax`, and `select` operations.
- **Analyzed Logical Plan:** Spark resolves the columns and confirms that `account_id` is a string and `tax` is a double.
- **Optimized Logical Plan:** Spark removes the unnecessary `amount` column from the final projection and adds an `isnotnull(amount)` condition to the filter.
- **Physical Plan:** Spark executes the query using `Scan ExistingRDD`, `Filter`, and `Project` operators.

The key distinction is:

**Logical Plan = what Spark needs to do**  
**Physical Plan = how Spark will execute it**

---------------------------------------------------------
## 6.      Document before/after runtime, resources, and DAG.


## Performance Summary

In [70]:
# A9 performance summary
performance_results = [
    ("Baseline aggregation", baseline_time),
    ("Salted aggregation", salted_time),
    ("AQE OFF", aqe_off_time),
    ("AQE ON", aqe_on_time),
    ("Shuffle join", shuffle_join_time),
    ("Broadcast join", broadcast_join_time),
    ("Shuffle partitions = 8", partitions_8_time),
    ("Shuffle partitions = 32", partitions_32_time),
    ("Shuffle partitions = 200", partitions_200_time),
    ("Cache materialization", cache_materialization_time)
]

performance_df = spark.createDataFrame(
    performance_results,
    ["experiment", "runtime_seconds"]
)

performance_df.show(truncate=False)

+------------------------+------------------+
|experiment              |runtime_seconds   |
+------------------------+------------------+
|Baseline aggregation    |3.566843600012362 |
|Salted aggregation      |4.102179000037722 |
|AQE OFF                 |16.48238439997658 |
|AQE ON                  |13.24765400000615 |
|Shuffle join            |15.36256159999175 |
|Broadcast join          |13.878071499988437|
|Shuffle partitions = 8  |7.009947200014722 |
|Shuffle partitions = 32 |5.747918900044169 |
|Shuffle partitions = 200|5.935323600017    |
|Cache materialization   |3.4279487999738194|
+------------------------+------------------+



## Record the resources

In [71]:
# Record local Spark environment details
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)
print(
    "AQE enabled:",
    spark.conf.get("spark.sql.adaptive.enabled")
)

Spark version: 4.2.0
Master: local[*]
Default parallelism: 8
Shuffle partitions: 200
AQE enabled: true


Spark version
    → Spark software version used

Master
    → Where Spark is running
      (in our case local[*])

Default parallelism
    → Default level of parallelism available locally

Shuffle partitions
    → Current SQL shuffle partition setting

AQE
    → Whether Adaptive Query Execution is enabled

## DAG evidence

Already captured Spark UI stage screenshots during the experiments